<a href="https://colab.research.google.com/github/internshipinabook/nlp-internship-in-a-book/blob/main/notebooks/week3_finding_topics_STARTER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ── Colab setup (skip if running locally) ──────────────────────
import os
if not os.path.exists('requirements.txt'):
    !git clone https://github.com/internshipinabook/nlp-internship-in-a-book.git book
    %cd book
    !pip install -r requirements.txt -q
    print('Colab setup complete ✅')
else:
    print('Running locally ✅')


# Week 3 — Finding the Topics (STARTER)
### The NLP Internship | LinguaAI Labs

This week: TF-IDF keyword extraction and LDA topic modelling on 50,000 unlabelled tickets.

**Fill in every `# YOUR CODE HERE` block.**

In [ ]:
import sys, subprocess, importlib
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
np.random.seed(42)
for pkg in ["scikit-learn","nltk","langdetect"]:
    try: importlib.import_module(pkg.replace("-","_"))
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install",pkg,"-q"])
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import nltk
for r in ["punkt","punkt_tab","stopwords","wordnet"]:
    nltk.download(r, quiet=True)
print("Setup complete ✅")

## Part 1 — TF-IDF Keywords by Channel

> ⏸️ **Pause and Predict**
> Before running: which top TF-IDF keyword for the social channel do you expect would NOT appear in the email channel's top keywords?

In [ ]:
import pandas as pd, os

# Smart loader: local package first → GitHub → sample data fallback
LOCAL_PATH = '../data/support_tickets.csv'
REMOTE_URL = 'https://raw.githubusercontent.com/internshipinabook/nlp-internship-in-a-book/main/data/support_tickets.csv'

df = None
if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
    print(f'Loaded from local package ✅')
else:
    try:
        df = pd.read_csv(REMOTE_URL)
        print(f'Loaded from GitHub repository ✅')
    except Exception as e:
        print(f'⚠️  Dataset not yet online ({e})')
        print('Generating sample data — upload support_tickets.csv to data/ folder')
        import numpy as np; np.random.seed(42)
        tickets = [
            'My account login is not working, please help',
            'I was charged twice for my subscription this month',
            'The app crashes whenever I try to upload a file',
            'How do I cancel my subscription?',
            'Great service, very happy with the product!',
            'I cannot reset my password, the email never arrives',
            'The dashboard is loading very slowly today',
            'I need a refund for my last order',
            'Feature request: can you add dark mode?',
            'Error 500 when trying to export my data',
        ] * 50
        np.random.shuffle(tickets)
        df = pd.DataFrame({
            'ticket_id':  range(1, 501),
            'text':       tickets[:500],
            'category':   np.random.choice(['billing','technical','account','general'], 500),
            'sentiment':  np.random.choice(['positive','negative','neutral'], 500, p=[0.3,0.4,0.3]),
            'priority':   np.random.choice(['low','medium','high'], 500),
        })
        print('Sample dataset ready (500 tickets) ✅')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()


## Part 2 — LDA Topic Model

The TF-IDF formula:
```
TF-IDF(w, d) = TF(w, d) × log( N / (1 + DF(w)) )
```

In [ ]:
texts = df["text_clean"].fillna("").tolist()

# LDA uses raw counts, not TF-IDF
lda_vec = CountVectorizer(
    max_features=3000, stop_words="english",
    min_df=10, max_df=0.85
)
dtm = lda_vec.fit_transform(texts)
vocab = lda_vec.get_feature_names_out()

print(f"Document-term matrix: {dtm.shape}")
print(f"Sparsity: {1 - dtm.nnz / (dtm.shape[0]*dtm.shape[1]):.1%}")

print("\nTraining LDA (8 topics, ~3 minutes)...")
lda = LatentDirichletAllocation(
    n_components=8, random_state=42,
    max_iter=20, learning_method="online"
)
lda.fit(dtm)
print(f"Log-likelihood: {lda.score(dtm):.1f}")

## Part 3 — Inspect and Name Topics

In [ ]:
# Print top words per topic
def print_topics(model, vocab, n_words=15):
    for i, topic in enumerate(model.components_):
        top_words = [vocab[j] for j in topic.argsort()[::-1][:n_words]]
        print(f"\nTopic {i+1}:")
        print(f"  {' | '.join(top_words[:8])}")
        print(f"  {' | '.join(top_words[8:])}")

print_topics(lda, vocab)

# YOUR CODE HERE — name each topic after reading the top words
TOPIC_NAMES = {
    0: "YOUR NAME HERE",
    1: "YOUR NAME HERE",
    2: "YOUR NAME HERE",
    3: "YOUR NAME HERE",
    4: "YOUR NAME HERE",
    5: "YOUR NAME HERE",
    6: "YOUR NAME HERE",
    7: "YOUR NAME HERE",
}

In [ ]:
# Assign dominant topic to each ticket
doc_topic_probs = lda.transform(dtm)
df["dominant_topic"] = doc_topic_probs.argmax(axis=1)
df["topic_confidence"] = doc_topic_probs.max(axis=1)
df["topic_name"] = df["dominant_topic"].map(TOPIC_NAMES)

print("Topic distribution:\n")
for topic_id, name in TOPIC_NAMES.items():
    count = (df["dominant_topic"] == topic_id).sum()
    conf  = df[df["dominant_topic"]==topic_id]["topic_confidence"].mean()
    print(f"  {name:<25}: {count:>6,} ({count/len(df):.1%}) avg conf={conf:.2f}")

## 🏆 Challenge Task

> Run LDA with 5, 8, and 12 topics. For each, compute average coherence. Is there a clear elbow?
> In a markdown cell: which number of topics would you choose — and why can't coherence alone decide?

## ⚠️ Common Mistakes

| Mistake | What goes wrong | Fix |
|---------|-----------------|-----|
| Taking LDA topic labels at face value | LDA gives you word distributions, not topic names. You have to interpret and label each topic yourself — the model cannot do that. | Read the top 10 words for each topic and write a one-sentence description |
| Choosing the number of topics by default | 5 is not the right answer for every corpus. Too few topics merge distinct themes. Too many split coherent ones. | Try 3, 5, 8, and 12 topics. Plot coherence scores and choose the elbow. |

## ✅ What You Can Do After This Week

- Explain TF-IDF and compute keyword profiles per channel
- Train an LDA topic model and evaluate topic coherence
- Name topics from top words and representative tickets
- Write a plain-language topic report for a non-technical client

---
## ✅ Week 3 Complete
**Next:** `week4_communication_STARTER.ipynb`

---
*Add `week3_finding_topics_STARTER.ipynb` to your internship portfolio.*

*The NLP Internship · LinguaAI Labs · github.com/InternshipInABook*

## ✅ By completing Week 3, you can now:

- Run LDA and interpret the output — not just describe it
- Label each topic with a one-sentence human-readable description
- Choose the number of topics deliberately using coherence scores
- Explain the difference between topic probability and topic assignment

📤 **GitHub:** Push week3_finding_topics_STARTER.ipynb and topic_labels.md. Commit: "Week 3: LDA topics labelled"


---

## 📚 Get the Full Book

This notebook is part of **The NLP Internship** — Book 2 of the InternshipInABook™ Series.

The full book includes:
- Complete week-by-week narrative and explanations
- All STOP AND TRACE code walkthroughs
- Fairness briefs, model cards, and deployment guides
- Certificate of Completion

👉 **[Get the book on Selar →](https://selar.com/8440iqfr61)**

---
*InternshipInABook™ Series · © Sakinat Folorunso 2026 · [github.com/internshipinabook](https://github.com/internshipinabook)*
